# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /Users/Santu/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/Santu/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Use-Case Data!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '22f4a2'. Skipping!
Property 'summary' already exists in node 'd9faff'. Skipping!
Property 'summary' already exists in node 'd7530d'. Skipping!
Property 'summary' already exists in node '51a8c6'. Skipping!
Property 'summary' already exists in node '3a9fbe'. Skipping!
Property 'summary' already exists in node 'c12951'. Skipping!
Property 'summary' already exists in node 'b5f1fe'. Skipping!
Property 'summary' already exists in node 'cad87a'. Skipping!
Property 'summary' already exists in node 'fe6d4c'. Skipping!
Property 'summary' already exists in node 'cb058a'. Skipping!
Property 'summary' already exists in node '146a47'. Skipping!
Property 'summary' already exists in node 'c93a33'. Skipping!
Property 'summary' already exists in node '496387'. Skipping!
Property 'summary' already exists in node '2d0275'. Skipping!
Property 'summary' already exists in node 'ea74ff'. Skipping!
Property 'summary' already exists in node '5cfbfb'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'c93a33'. Skipping!
Property 'summary_embedding' already exists in node '22f4a2'. Skipping!
Property 'summary_embedding' already exists in node 'fe6d4c'. Skipping!
Property 'summary_embedding' already exists in node 'cad87a'. Skipping!
Property 'summary_embedding' already exists in node 'c12951'. Skipping!
Property 'summary_embedding' already exists in node 'd9faff'. Skipping!
Property 'summary_embedding' already exists in node 'b5f1fe'. Skipping!
Property 'summary_embedding' already exists in node 'd7530d'. Skipping!
Property 'summary_embedding' already exists in node '51a8c6'. Skipping!
Property 'summary_embedding' already exists in node 'ea74ff'. Skipping!
Property 'summary_embedding' already exists in node '3a9fbe'. Skipping!
Property 'summary_embedding' already exists in node '146a47'. Skipping!
Property 'summary_embedding' already exists in node '496387'. Skipping!
Property 'summary_embedding' already exists in node '2d0275'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 86, relationships: 713)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 86, relationships: 713)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [11]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [12]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.


✅ Answer #1:


**1. SingleHopSpecificQuerySynthesizer** - Creates easy, direct questions that need only one piece of information to answer.  

**2. MultiHopAbstractQuerySynthesizer** - Creates complex questions that need multiple pieces of information and abstract thinking to answer.  

**3. MultiHopSpecificQuerySynthesizer** - Creates complex questions that need multiple specific facts from different parts of the documents to answer.  

Finally, we can use our `TestSetGenerator` to generate our testset!

In [13]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/10 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,How does the Copilot feature assist users in t...,[Introduction ChatGPT launched in November 202...,The provided context does not include specific...,single_hop_specifc_query_synthesizer
1,How does OpenAI's technology influence the usa...,[Table 1: ChatGPT daily message counts (millio...,The context indicates that OpenAI's technology...,single_hop_specifc_query_synthesizer
2,What is SOC in relation to ChatGPT usage?,[Variation by Occupation Figure 23 presents va...,Variation by Occupation Figure 23 presents var...,single_hop_specifc_query_synthesizer
3,How does the 2.5 billion messages per day refl...,[Conclusion This paper studies the rapid growt...,"By July 2025, ChatGPT users collectively sent ...",single_hop_specifc_query_synthesizer
4,how activities like editing critiquing transla...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,the context shows that most chatgpt usage invo...,multi_hop_abstract_query_synthesizer
5,wht work-messages share and ask vs do diffrenc...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The context shows that users in highly paid pr...,multi_hop_abstract_query_synthesizer
6,Wht are the diffrences in mesage types (Asking...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The context indicates that users in highly pai...,multi_hop_abstract_query_synthesizer
7,how OpenAI ChatGPT used for openAI and how man...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"The context explains that ChatGPT, developed b...",multi_hop_specific_query_synthesizer
8,Whn in Novmber 2022 was ChatGPT launced and ho...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"ChatGPT was launched in November 2022, and by ...",multi_hop_specific_query_synthesizer
9,How do the details in Appendix A and Appendix ...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The information in Appendix A provides an over...,multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '116de4'. Skipping!
Property 'summary' already exists in node '6cc941'. Skipping!
Property 'summary' already exists in node '922807'. Skipping!
Property 'summary' already exists in node 'd2b4c2'. Skipping!
Property 'summary' already exists in node 'f80960'. Skipping!
Property 'summary' already exists in node '0715ad'. Skipping!
Property 'summary' already exists in node '56c168'. Skipping!
Property 'summary' already exists in node '4fed38'. Skipping!
Property 'summary' already exists in node 'ef8e5a'. Skipping!
Property 'summary' already exists in node 'b755e0'. Skipping!
Property 'summary' already exists in node '9dfb72'. Skipping!
Property 'summary' already exists in node '66c140'. Skipping!
Property 'summary' already exists in node '32f843'. Skipping!
Property 'summary' already exists in node 'cf5c4b'. Skipping!
Property 'summary' already exists in node '9ad121'. Skipping!
Property 'summary' already exists in node '5ca2fc'. Skipping!
Property

...

Property 'summary_embedding' already exists in node 'ef8e5a'. Skipping!
Property 'summary_embedding' already exists in node '922807'. Skipping!
Property 'summary_embedding' already exists in node '0715ad'. Skipping!
Property 'summary_embedding' already exists in node '32f843'. Skipping!
Property 'summary_embedding' already exists in node '56c168'. Skipping!
Property 'summary_embedding' already exists in node '116de4'. Skipping!
Property 'summary_embedding' already exists in node '6cc941'. Skipping!
Property 'summary_embedding' already exists in node 'f80960'. Skipping!
Property 'summary_embedding' already exists in node '4fed38'. Skipping!
Property 'summary_embedding' already exists in node 'b755e0'. Skipping!
Property 'summary_embedding' already exists in node 'd2b4c2'. Skipping!
Property 'summary_embedding' already exists in node '9dfb72'. Skipping!
Property 'summary_embedding' already exists in node '66c140'. Skipping!
Property 'summary_embedding' already exists in node '9ad121'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [15]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,"Who is Bick et al., 2024?",[Introduction ChatGPT launched in November 202...,"Bick et al., 2024 is a referenced work in the ...",single_hop_specifc_query_synthesizer
1,Jun 2025 how many messages non work,[Month Non-Work (M) (%) Work (M) (%) Total Mes...,"In June 2025, there were 1,911 million non-wor...",single_hop_specifc_query_synthesizer
2,What is Handa in the context of message counts...,[Total daily counts are exact measurements of ...,The context does not provide a specific defini...,single_hop_specifc_query_synthesizer
3,What is the Sectin in the context of ChatGPT u...,[Variation by Occupation Figure 23 presents va...,The provided context does not define or explai...,single_hop_specifc_query_synthesizer
4,Based on the observed growth and distribution ...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The data indicates that while total message vo...,multi_hop_abstract_query_synthesizer
5,so like how does work activities and tasks rel...,[<1-hop>\n\nVariation by Occupation Figure 23 ...,The context explains that ChatGPT usage varies...,multi_hop_abstract_query_synthesizer
6,how much chatgpt messages are used for practic...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,"In June 2024, the total ChatGPT messages amoun...",multi_hop_abstract_query_synthesizer
7,H0w does the global economic impact of ChatGPT...,[<1-hop>\n\nConclusion This paper studies the ...,The global economic impact of ChatGPT is close...,multi_hop_abstract_query_synthesizer
8,How did the rapid growth of ChatGPT since its ...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"Since its launch in November 2022, ChatGPT exp...",multi_hop_specific_query_synthesizer
9,Based on the findings of Collis and Brynjolfss...,[<1-hop>\n\nTotal daily counts are exact measu...,The second context provides detailed evidence ...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [17]:
from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE8"

# Use existing dataset instead of creating new one
langsmith_dataset = client.read_dataset(dataset_name=dataset_name)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [18]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [19]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [20]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [21]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [22]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [23]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [24]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [25]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [26]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [27]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

'Based on the provided context, people are using AI, particularly generative AI like ChatGPT, in a variety of ways both at work and outside of work. Specifically:\n\n- AI is performing workplace tasks that either augment or automate human labor.\n- Users seek to produce digital products such as writing, software code, and spreadsheets using generative AI.\n- People use AI with different intentions, including asking for information or advice, doing tasks, or expressing themselves.\n- Use cases of generative AI include therapy/companionship, games and role play, relationships and personal reflection, and self-expression.\n- AI serves roles as co-workers producing output or as co-pilots improving human problem-solving.\n- There is evidence of widespread usage encompassing professional, educated users in high-paid occupations as well as everyday users.\n\nIn summary, people are using AI for workplace automation and augmentation, creating various digital outputs, seeking advice, engaging in

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [28]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [29]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`: Evaluates if the answer is factually correct and relevant to the question
- `labeled_helpfulness_evaluator`: Evaluates if the answer is helpful to the user, considering the correct reference answer
- `dopeness_evaluator`: Different evaluators test different aspects of answer quality - correctness, helpfulness, and engagement

## LangSmith Evaluation

In [30]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'worthwhile-offer-39' at:
https://smith.langchain.com/o/56330a9f-2636-4f7a-9285-2dd45c07515b/datasets/2b06fb82-17df-4cb4-bed8-ed78577e0ff8/compare?selectedSessions=6ced97cd-5df6-44fb-95ea-cc73ff002457




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,Based on the rapid growth of ChatGPT usage fro...,"Based on the provided context, the rapid growt...",None,"The data indicates that by July 2025, ChatGPT ...",1,1,0,7.030425,01ff7b8d-01ac-484a-bc7e-a12eb21b492b,33acfeab-aa44-4ac7-979c-b124dc92836e
1,how many messages in July 2025 compared to Jun...,"In June 2025, there were 2,627 million (2.627 ...",None,"In June 2025, total messages were 2,627 millio...",0,0,0,4.806730,4c6e4f33-cb68-424f-8a29-7d791b1115e8,0ce23e29-0861-4816-b8f4-d9ad8599bc2a
2,Based on the findings of Collis and Brynjolfss...,The analysis in the second context supports an...,None,The second context provides detailed evidence ...,1,1,0,6.958633,d9e96332-ad34-49fb-ac18-544d71a741b7,bec7ba93-2970-420a-8c1b-6ecc0f59f810
3,How did the rapid growth of ChatGPT since its ...,"According to the context provided, ChatGPT exp...",None,"Since its launch in November 2022, ChatGPT exp...",1,0,0,3.666287,629e3ae5-1b08-4160-9000-27f3290bf6ec,df201319-3522-4fdb-b593-a6945ecd0a56
4,H0w does the global economic impact of ChatGPT...,The global economic impact of ChatGPT relates ...,None,The global economic impact of ChatGPT is close...,1,1,0,6.997263,6d26ebb5-039f-41a0-b55f-b4f86281e555,1cc67ce2-e6cb-4c6a-bfa5-9fb0d174ffc6
5,how much chatgpt messages are used for practic...,Based on the provided context:\n\n- In June 20...,None,"In June 2024, the total ChatGPT messages amoun...",1,0,0,12.489845,d2c63f87-6286-4901-8069-4b8378d1b244,42043e0f-41be-4e02-8d66-f8e5c36c4dd2
6,so like how does work activities and tasks rel...,"Based on the provided context, here is how wor...",None,The context explains that ChatGPT usage varies...,1,1,0,14.407108,c55fee36-3c10-46f2-95f0-000ca0176277,91895d51-63af-4ce4-a2ce-c131086fcc75
7,Based on the observed growth and distribution ...,"Based on the provided context, the shift in Ch...",None,The data indicates that while total message vo...,1,1,0,8.392794,00da30f2-adab-4cf0-a5f6-c41d5c713800,ceab4da8-48e6-4f6c-bce0-f34788893415
8,What is the Sectin in the context of ChatGPT u...,The section in the context of ChatGPT usage by...,None,The provided context does not define or explai...,0,0,0,0.919075,b8fd6790-8be8-4bbf-84a1-efa061e92af3,55d27f34-74be-46c4-9688-8c55b2d7e09f
9,What is Handa in the context of message counts...,Handa refers to a study or methodology (Clio m...,None,The context does not provide a specific defini...,0,0,0,2.199513,7c322aee-d478-42c7-8424-49445f5afc43,fa6801c7-d0ec-4c3f-a9a0-70f8968408de


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [31]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [32]:
rag_documents = docs

In [33]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

✅ Answer #2:

Actually chuck size has seceral impacts on the performance of our application.

- **Retrieval Quality**: 
Smaller chunks give more precise matches but may miss context. Larger chunks give more context but less precise matches.
- **Information Completeness**: Smaller chunks may split important information across chunks. Larger chunks are more likely to contain complete answers.
- **Computational Cost**: Smaller chunks mean more chunks to process and higher embedding costs. Larger chunks mean fewer chunks and lower embedding costs.
- **Answer Quality**: Smaller chunks work better for specific, focused questions. Larger chunks work better for complex questions that need context.



In [34]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

✅ Answer #3:

Changing / moifying the embedding model can change the application performance

- Modifying our embedding model would affect performance because better embedding models create more accurate vector representations of text, which leads to better semantic matching.   
- Higher-quality embeddings help find more relevant chunks for a given question, improving answer relevance. Advanced embedding models better understand context, synonyms, and meaning, which reduces false matches.   
- However, more powerful embedding models may be slower and more expensive but provide better results.


In [35]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [36]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [37]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [38]:
dopeness_rag_chain.invoke({"question" : "How are people using AI to make money?"})

'Yo, here’s the lowdown straight from the AI streets of the future: People aren’t just using AI as a slick tool to *do* tasks—they’re leaning heavy on it as their ultimate sidekick for *brainpower*. ChatGPT is flexing as an advisor and research assistant, helping knowledge workers level up their decision-making game. Imagine AI as your own personal Yoda, boosting productivity by fueling smarter moves, not just grinding through work like a robot.\n\nThe juice is in using AI for decision support in knowledge-intensive gigs, where better choices = bigger wins. So, people are cashing in by integrating generative AI into their workflow—augmenting their skills, turbocharging research, and making smarter calls that flip into higher output and, naturally, more dough. The bang? It’s estimated users get a trillion-dollar+ surplus vibe because of this AI wizardry, meaning money flows when humans and AI team up like a dream.\n\nIn the most rad nutshell: AI’s the secret sauce making knowledge worke

Finally, we can evaluate the new chain on the same test set!

In [39]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'roasted-event-66' at:
https://smith.langchain.com/o/56330a9f-2636-4f7a-9285-2dd45c07515b/datasets/2b06fb82-17df-4cb4-bed8-ed78577e0ff8/compare?selectedSessions=d45b74e1-9c12-449f-9245-09a1d19ec420




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,Based on the rapid growth of ChatGPT usage fro...,"Alright, buckle up for a wild ride through the...",None,"The data indicates that by July 2025, ChatGPT ...",1,1,1,11.234428,01ff7b8d-01ac-484a-bc7e-a12eb21b492b,6f70e3cf-4e8a-4598-aba8-bbdbc7ebe13f
1,how many messages in July 2025 compared to Jun...,"Alright, check this out — in June 2025, ChatGP...",None,"In June 2025, total messages were 2,627 millio...",1,0,1,3.586137,4c6e4f33-cb68-424f-8a29-7d791b1115e8,27105551-8009-4352-b195-4339a995f5f3
2,Based on the findings of Collis and Brynjolfss...,"Alright, strap in for some next-level economic...",None,The second context provides detailed evidence ...,1,1,1,6.613758,d9e96332-ad34-49fb-ac18-544d71a741b7,380c3370-7b59-44d9-810a-532c8880d55e
3,How did the rapid growth of ChatGPT since its ...,"Alright, strap in tight—because the story of C...",None,"Since its launch in November 2022, ChatGPT exp...",1,1,1,11.446797,629e3ae5-1b08-4160-9000-27f3290bf6ec,edc4a98f-c7c0-499e-8cb7-e87d13195503
4,H0w does the global economic impact of ChatGPT...,"Alright, buckle up, because here’s the lowdown...",None,The global economic impact of ChatGPT is close...,1,1,1,7.323583,6d26ebb5-039f-41a0-b55f-b4f86281e555,bba9a3a3-4f75-4ba6-94f6-6eb854668427
5,how much chatgpt messages are used for practic...,"Alright, time to drop some seriously slick ins...",None,"In June 2024, the total ChatGPT messages amoun...",0,0,1,9.721152,d2c63f87-6286-4901-8069-4b8378d1b244,c8dbad66-00e2-40b7-bfb0-4a6f16e71b4f
6,so like how does work activities and tasks rel...,"Alright, buckle up—let’s dive deep into the AI...",None,The context explains that ChatGPT usage varies...,1,1,1,10.453476,c55fee36-3c10-46f2-95f0-000ca0176277,048da618-9a5b-4df4-8c27-e37fa8af37dc
7,Based on the observed growth and distribution ...,"Alright, buckle up—this is where the AI story ...",None,The data indicates that while total message vo...,1,1,1,9.727808,00da30f2-adab-4cf0-a5f6-c41d5c713800,938de726-7cd4-4733-9381-5e39a72d0685
8,What is the Sectin in the context of ChatGPT u...,"Oh snap, you're diving into the magic behind h...",None,The provided context does not define or explai...,0,0,1,2.957790,b8fd6790-8be8-4bbf-84a1-efa061e92af3,dcc84a44-cac4-4a3b-a9f5-f4c1fe55548f
9,What is Handa in the context of message counts...,"Alright, let’s rock this answer with max dopen...",None,The context does not provide a specific defini...,0,0,1,3.651735,7c322aee-d478-42c7-8424-49445f5afc43,1d108a4c-55f8-4095-963f-118183245deb


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.